# Agentic Image Generation

`01_Foundations/00_Theory_and_Foundations/Model_Landscape_and_Hugging_Face/02_Diffusers/`
covers image-generation *models*: you hold a prompt, you call a pipeline, you get pixels.
The interesting decisions are yours.

This notebook covers something else — **an agent deciding mid-conversation to produce an
artefact**, writing its own prompt for it, and handing back bytes. Three things change when
generation moves inside the loop:

1. **The prompt is written by the model**, from conversation context you never see turned
   into words. It is now something to inspect.
2. **The agent must also decide *not* to generate.** A tool that always fires is not a tool.
3. **There is no ground truth.** `09_Hosted_Code_Execution.ipynb` could check the model's
   arithmetic against `statistics.stdev`. There is no `statistics` for "is this label
   right", and — as this notebook shows on a real run — the agent will describe its own
   output as correct without ever looking at it.

That third point is the one that makes this a distinct pattern rather than a fourth hosted
tool.

## Learning objectives

1. Attach `image_generation` and read back the prompt the model wrote for itself.
2. Show the same tool staying silent when the turn does not call for an artefact.
3. Demonstrate that the agent's narration of its own artefact is **not evidence**, by
   feeding the artefact back to a vision model.
4. Close the loop — a refinement turn, and watch the call's `action` flip from `generate`
   to `edit` on its own.
5. Place artefact-producing tools against retrieval (`07_`) and computation (`09_`).

## Where this fits

- `07_Hosted_vs_Client_Side_Tools.ipynb` — the hosted/client split. Read first.
- `09_Hosted_Code_Execution.ipynb` — hosted tools differ in how much they let you see.
  This notebook adds a third point to that table.
- `08_Vision_Driven_Computer_Use.ipynb` — the vision-input mechanism used here for QA is
  the one used there for control.
- `4. Workflow_Pattern/` evaluator-optimizer — section 4 is that pattern with a vision
  model as the evaluator, because the thing being evaluated is not text.

## Prerequisites and cost

`OPENAI_API_KEY` on an account with image generation enabled. **This notebook is the most
expensive in the folder.** Each generation is cents rather than fractions of a cent, and
measured latency on `gpt-image-1-mini` at `quality="low"` was **12.3s / 12.5s / 12.8s**
across three runs — roughly twenty times a text turn. There are four generations below.

In [ ]:
# ============ SETUP ============
import base64
import io
import time

from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from PIL import Image

load_dotenv()
client = OpenAI()

MODEL = "gpt-4.1-mini"          # the agent; it decides whether to call the tool

# Verified against the installed SDK (openai.types.responses.tool.ImageGeneration).
# The full field set is: action, background, input_fidelity, input_image_mask, model,
# moderation, output_compression, output_format, partial_images, quality, size.
# Kept small and cheap here; section 5 covers the rest.
IMAGE_TOOL = {
    "type": "image_generation",
    "model": "gpt-image-1-mini",
    "quality": "low",
    "size": "1024x1024",
}


def show(call, max_px=420):
    """Decode an image_generation_call's base64 result, display it, report its size."""
    raw = base64.b64decode(call.result)
    img = Image.open(io.BytesIO(raw))
    print(f"  {len(raw):,} bytes · {img.size[0]}x{img.size[1]} · {img.mode}")
    # You own these bytes. Persisting them is your problem:
    #   pathlib.Path("label.png").write_bytes(raw)
    display(img.copy().resize((max_px, max_px)))
    return img

## 1. The agent decides, and writes its own prompt

Nothing below names an image size, a style, or a subject in prompt form. The request is a
business one. The model has to notice that an artefact is the right response, then compose
the generation prompt itself.

In [ ]:
# ============ A TURN WHERE GENERATING IS THE RIGHT MOVE ============
REQUEST = (
    "Our roastery, Ninth Street, sells a single-origin Ethiopian. "
    "Design a square label for the bag and show it to me."
)

t0 = time.time()
generated = client.responses.create(model=MODEL, input=REQUEST, tools=[IMAGE_TOOL])
print(f"  {time.time() - t0:.1f}s")
print(f"  output items: {[i.type for i in generated.output]}")

call = next(i for i in generated.output if i.type == "image_generation_call")
print(f"  status: {call.status}\n")
img = show(call)

In [ ]:
# ============ THE PROMPT IT WROTE FOR ITSELF ============
# revised_prompt is not in ImageGenerationCall.model_fields — the SDK model accepts extra
# fields and the API populates it, so reach for it defensively.
print("revised_prompt:")
print(f"  {getattr(call, 'revised_prompt', None)}\n")

print("what the call actually ran with:")
for field in ("action", "size", "quality", "output_format", "background"):
    print(f"  {field:<14} {getattr(call, field, '—')}")

print(f"\nusage.output_tokens = {generated.usage.output_tokens}")
print("  ...which counts the text reply only. The image is billed separately and is")
print("  invisible here — do not cost this tool from `usage`.")

### Why `revised_prompt` matters

`09_` established that hosted tools differ in how much they show you, with
`code_interpreter` returning the source it ran. **`revised_prompt` is the image tool's
equivalent** — the model's intent, in words, before any pixels existed.

That is the only auditable artefact in the whole call. When the output is wrong you need to
know whether the model misunderstood the *request* or the renderer misexecuted the
*prompt*, and those have different fixes: the first is a system-prompt problem, the second
is a model or quality-setting problem. Without `revised_prompt` you cannot tell them apart.

Two more things this cell establishes:

- **`result` is base64, not a URL.** Nobody is hosting this for you. Storage, delivery,
  retention and deletion are yours from the moment the response lands — which is a
  different operational position from `web_search` returning a citation.
- **`usage` undercounts.** The image does not appear in `output_tokens`. A cost dashboard
  built on `usage` alone will silently under-report an agent that generates images.

Add the third row to `09_`'s visibility table:

| Hosted tool | What comes back |
|---|---|
| `web_search` | a conclusion, and roughly what was searched |
| `code_interpreter` | the source it ran, stdout, and any files produced |
| `image_generation` | the **prompt it wrote**, the config it used, and the bytes |

## 2. Restraint is the other half of "mid-conversation"

A tool that fires on every turn is not a decision, it is a pipeline. The test for
*mid-conversation* is whether the model can leave the tool alone.

Same tool, same attachment. A request that is adjacent to imagery — it is explicitly about
a logo — but asks for advice, not an artefact.

In [ ]:
# ============ RESTRAINT: THE TOOL IS ATTACHED AND SHOULD NOT FIRE ============
restrained = client.responses.create(
    model=MODEL,
    input=(
        "We're designing a logo for a coffee roastery called Ninth Street. "
        "Before any visuals, what three questions should I answer first?"
    ),
    tools=[IMAGE_TOOL],
)

types = [i.type for i in restrained.output]
fired = "image_generation_call" in types
print(f"  output items: {types}")
print(f"  generated an image: {fired}   <- should be False\n")
print(restrained.output_text[:400])

### Reading that result

The word "logo" is present, the tool is attached, and the model still returns only a
message. The deciding signal is *"before any visuals"* — an instruction about sequencing,
not about images, and the model honoured it.

This is worth testing explicitly because the failure is quiet and expensive. An
over-eager image tool does not throw; it adds twelve seconds and a few cents to turns that
wanted a sentence, and the user experiences it as an agent that does not listen.

Where you steer it, in order of bluntness:

1. **System prompt** — "generate an image only when the user asks to see something."
2. **`tool_choice`** — `"none"` to forbid, `{"type": "image_generation"}` to force. Use
   this when the *turn type* is known to your application, not the model.
3. **Don't attach the tool.** If the current step of your graph cannot legitimately produce
   an image, the cheapest guardrail is an empty `tools` list.

Option 3 is the one that generalises: attaching tools per node rather than per agent is a
LangGraph habit worth keeping here.

## 3. The narration is not evidence

Here is where artefact tools diverge sharply from the other two hosted tools.

`09_` could grade the model against `statistics.stdev`. `07_`'s web search could be checked
against the source. **An image has no such oracle** — and the agent does not look at what it
produced. It writes a prompt, receives bytes, and then narrates what it *intended*.

The cell below closes that loop the only way available: send the artefact back in as an
input image and make a vision model read it.

In [ ]:
# ============ WHAT THE AGENT SAID IT MADE ============
print(generated.output_text[:400])

In [ ]:
# ============ WHAT IS ACTUALLY ON THE LABEL ============
audit = client.responses.create(
    model="gpt-4.1-mini",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": (
                "Transcribe EVERY piece of text visible in this label, exactly as "
                "rendered, including misspellings. Then state whether every string is a "
                "real, correctly spelled English word. Answer in two short lines."
            )},
            {"type": "input_image", "image_url": f"data:image/png;base64,{call.result}"},
        ],
    }],
)
print(audit.output_text)

### What just happened

On the run this notebook was written against, the agent's narration read:

> *"Here is a square label design for your Ninth Street Roastery single-origin Ethiopian
> coffee... the roastery's name at the top, 'Ethiopian Coffee' prominently in the centre..."*

and the audit transcribed:

> `NINTH STRFET` · `IUCICS AATRIS` · `BRIGHT IPRJITK TLORAL`

Confident narration, mangled artefact. Your run will differ in the specific corruption —
text rendering is where current image models are weakest — but the *structure* of the
failure reproduces: **the generating agent's description of its own output is written from
the prompt it sent, not from the pixels it got back.**

Three consequences worth carrying out of this notebook:

- **Never surface the narration as a description of the artefact.** It is a description of
  the request. Showing it next to a broken image is worse than showing the image alone.
- **The check must be a separate turn with vision input.** Not the same call, not the same
  model instance reasoning about what it "made" — a fresh look at the bytes.
- **Ask for a transcription, not a judgement.** "Does this look good?" gets agreement.
  "Transcribe every string, exactly as rendered" gets evidence, and the evidence is what
  your code can then assert on.

Structurally this is the **evaluator-optimizer** pattern from `4. Workflow_Pattern/`. The
only new part is that the evaluator has to be a vision model, because the artefact under
evaluation is not text. Everything else — generate, grade, feed the grade back — is a
pattern you already have.

## 4. The optimizer half: refining in place

Send the correction as a normal follow-up, chained with `previous_response_id`. Watch one
field in the result.

In [ ]:
# ============ THE REFINEMENT TURN ============
refined = client.responses.create(
    model=MODEL,
    previous_response_id=generated.id,
    input=(
        "The wording came out misspelled. Remove all text except the words NINTH STREET, "
        "and make the background cream instead of dark."
    ),
    tools=[IMAGE_TOOL],
)

call2 = next(i for i in refined.output if i.type == "image_generation_call")
print(f"  turn 1 action: {call.action}")
print(f"  turn 2 action: {call2.action}   <- flipped on its own")
print(f"\n  revised_prompt: {getattr(call2, 'revised_prompt', None)}\n")
img2 = show(call2)

### The `action` field is doing real work

Nothing in the follow-up said "edit". The tool config still says
`"model": "gpt-image-1-mini"` and nothing else. Yet `action` came back as **`edit`** rather
than `generate`, because `previous_response_id` put the earlier image in scope and the
model routed to the editing path itself.

That is a genuine convenience and a genuine trap:

- **Convenience** — multi-turn refinement is the default interaction for generated
  artefacts, and you get it without managing image state or re-uploading anything.
- **Trap** — the branch is invisible unless you read `action`. An "edit" that silently
  regenerated from scratch, or a "generate" that unexpectedly inherited the previous
  image's composition, are both real outcomes, and `action` is the only field that tells
  you which one you got. Log it.

The loop is now complete and it is the one you already know: **generate → verify with
vision → correct → verify again.** Re-running the section 3 audit against `call2.result` is
how you would close it; in production that assertion is the gate, not a print statement.

## 5. The configuration and cost surface

Verified against `openai.types.responses.tool.ImageGeneration` in the installed SDK.

| Field | Values | Why you would touch it |
|---|---|---|
| `model` | `gpt-image-1-mini`, `gpt-image-1`, `gpt-image-1.5`, `gpt-image-2`, … | The main cost/quality lever. `-mini` for iteration, full for delivery |
| `quality` | `low` · `medium` · `high` · `auto` | The second cost lever, and the one that most affects text rendering |
| `size` | `1024x1024` · `1024x1536` · `1536x1024` · `auto` | Pin it. `auto` makes cost non-deterministic |
| `action` | `generate` · `edit` · `auto` | Force the branch instead of letting `previous_response_id` decide |
| `background` | `transparent` · `opaque` · `auto` | `transparent` needs `png` or `webp` |
| `output_format` | `png` · `webp` · `jpeg` | `webp` + `output_compression` cuts the base64 you carry |
| `output_compression` | int | Only meaningful for `webp` / `jpeg` |
| `moderation` | `auto` · `low` | The content seam. See below |
| `partial_images` | int | Stream previews while generating — a latency-perception fix, not a latency fix |
| `input_image_mask` | mask object | Inpainting: edit a region, keep the rest |
| `input_fidelity` | `high` · `low` | How closely an `edit` preserves the source |

**On `moderation`.** This is the one field that is a policy decision rather than a
performance one, and it points at a risk the other hosted tools do not have. `web_search`
and `code_interpreter` return things that already existed or that you can re-derive. This
tool **manufactures content in your product's name**, and the request that produced it was
written by a model from a user's conversation. The user-facing prompt is therefore not the
thing to moderate — `revised_prompt` is. Log it, and if you moderate anything, moderate
that.

**On latency.** Twelve seconds is not a tuning problem, it is the shape of the tool. Design
the turn around it: generate on an explicit user action rather than speculatively, and use
`partial_images` if the wait needs to feel shorter. An agent that might spend twelve seconds
on any given turn needs a different UI from one that answers in half a second.

## 6. Where artefact tools sit

`07_` split tools by *who executes*. `09_` split hosted tools by *what you can see*. This
adds a third axis — **what you can verify**:

| | `web_search` | `code_interpreter` | `image_generation` |
|---|---|---|---|
| Produces | a retrieved claim | a computed value | a new artefact |
| Oracle available | the source | re-run the computation | **none** |
| How you check it | cite and compare | assert on the result | a second model, with vision |
| Check is | cheap, exact | cheap, exact | **costly, approximate** |
| Failure looks like | a wrong fact | a wrong number | a *confident description of something else* |

The last row is the one to remember. A wrong number is visibly a number that is wrong. A
mangled label arrives wrapped in a fluent paragraph explaining how good it is.

**Reach for it when** the artefact is the deliverable and a human will look at it before it
matters — mockups, drafts, illustrations inside a conversation.

**Do not reach for it when** the output ships unreviewed, must contain exact text, or
carries your brand. Text rendering is the current failure mode, and "the label says the
right words" is not something you can assert cheaply.

## Key takeaways

1. **The model writes the generation prompt.** `revised_prompt` is the auditable artefact
   of the call — it is how you tell a misunderstood *request* from a misexecuted *prompt*.
2. **`result` is base64, not a URL.** You own storage, delivery and retention the moment
   the response lands.
3. **`usage` does not count the image.** Cost dashboards built on `usage` under-report
   image-generating agents.
4. **Restraint is half the pattern.** Test that the tool stays silent; an over-eager image
   tool fails quietly, at twelve seconds and cents a turn. The cheapest control is not
   attaching it on nodes that cannot legitimately produce an image.
5. **The agent's narration describes the prompt, not the pixels.** Verified on a live run:
   a confident description of a label reading `NINTH STRFET`. Never surface it as a
   description of the artefact.
6. **Verification needs a second turn with vision input**, and must ask for a
   *transcription* rather than a judgement — evidence your code can assert on.
7. **`action` flips to `edit` by itself** when `previous_response_id` is set. Convenient,
   and invisible unless you log it.
8. **Artefact tools have no oracle.** That, not hosting, is what separates this from
   `07_` and `09_`: checking is costly and approximate, and the failure arrives
   well-described.

### Next

- `08_Vision_Driven_Computer_Use.ipynb` — the same vision input, used to act rather than
  to audit.
- `4. Workflow_Pattern/` evaluator-optimizer — the pattern section 3 and 4 instantiate.